In [ ]:
import pandas as pd

In [ ]:
pip install transformers accelerate torch torchvision pillow

In [ ]:
from transformers import AutoProcessor, AutoModelForImageTextToText
import torch
import torch.nn.functional as F

In [ ]:
df_questions = pd.read_csv("df_questions_2 (1).csv")

In [ ]:
df_questions.head()

,tag,image_name,image_data_url,true_description,description_for_answer,description_for_question,false_source_image_name,generated_question,used_prompt
0,concurrence,882048099.jpg,"data:image/jpeg;base64,/9j/4AAQSkZJRgABAQEAtAC...",A woman in a blue bathing suit and straw hat ...,A woman in a blue bathing suit and straw hat ...,A woman in a blue bathing suit and straw hat ...,NaN,Is there a woman in a blue bathing suit and st...,You need to formulate a single binary (yes/no)...
1,false_detail,882048099.jpg,"data:image/jpeg;base64,/9j/4AAQSkZJRgABAQEAtAC...",A woman in a blue bathing suit and straw hat ...,A woman in a gray bathing suit and straw hat ...,A woman in a gray bathing suit and straw hat ...,882048099.jpg,What color is the bathing suit the woman is we...,You are given two descriptions of the same ima...
2,image,882048099.jpg,"data:image/jpeg;base64,/9j/4AAQSkZJRgABAQEAtAC...",A woman in a blue bathing suit and straw hat ...,Four women in all black barefoot posing on th...,A woman in a blue bathing suit and straw hat ...,5922945248.jpg,Is there a woman in a blue bathing suit and st...,You need to formulate a single binary (yes/no)...
3,text,882048099.jpg,"data:image/jpeg;base64,/9j/4AAQSkZJRgABAQEAtAC...",A woman in a blue bathing suit and straw hat ...,Four women in all black barefoot posing on th...,Four women in all black barefoot posing on th...,5922945248.jpg,Are there women posing on the beach?,You need to formulate a single binary (yes/no)...
4,concurrence,199413509.jpg,"data:image/jpeg;base64,/9j/4AAQSkZJRgABAQEAtAC...",A woman peers out from her halfway opened blu...,A woman peers out from her halfway opened blu...,A woman peers out from her halfway opened blu...,NaN,Is there a woman peeking out from a blue door?,You need to formulate a single binary (yes/no)...


In [ ]:
db_fd = df_questions[df_questions["tag"] == "false_detail"]

In [ ]:
db_con = df_questions[df_questions["tag"] == "concurrence"]

In [ ]:
db_i = df_questions[df_questions["tag"] == "image"]

In [ ]:
db_t = df_questions[df_questions["tag"] == "text"]

Запускаем модель локально. Для всех сценариев использовали Qwen3-VL-2B-Instruct, а также дополнительно для false_detail использовали ещё одну модель Qwen3-VL-4B-Instruct, чтобы посмотреть улучшатся ли результаты ответов модели с большим количеством параметров.

In [ ]:
processor = AutoProcessor.from_pretrained("Qwen/Qwen3-VL-2B-Instruct")
HF_TOKEN = ""

processor = AutoProcessor.from_pretrained(
    "Qwen/Qwen3-VL-2B-Instruct",
    token=HF_TOKEN
)

model = AutoModelForImageTextToText.from_pretrained(
    "Qwen/Qwen3-VL-2B-Instruct",
    token=HF_TOKEN,
    torch_dtype=torch.bfloat16,   # экономия памяти GPU
    device_map="cuda"             # cuda - явно на GPU; auto если несколько GPU или нет уверенности
)
model.eval()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


preprocessor_config.json:   0%|          | 0.00/390 [00:00<?, ?B/s]

chat_template.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

...

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/4.26G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/625 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/269 [00:00<?, ?B/s]

Qwen3VLForConditionalGeneration(
  (model): Qwen3VLModel(
    (visual): Qwen3VLVisionModel(
      (patch_embed): Qwen3VLVisionPatchEmbed(
        (proj): Conv3d(3, 1024, kernel_size=(2, 16, 16), stride=(2, 16, 16))
      )
      (pos_embed): Embedding(2304, 1024)
      (rotary_pos_emb): Qwen3VLVisionRotaryEmbedding()
      (blocks): ModuleList(
        (0-23): 24 x Qwen3VLVisionBlock(
          (norm1): LayerNorm((1024,), eps=1e-06, elementwise_affine=True)
          (norm2): LayerNorm((1024,), eps=1e-06, elementwise_affine=True)
          (attn): Qwen3VLVisionAttention(
            (qkv): Linear(in_features=1024, out_features=3072, bias=True)
            (proj): Linear(in_features=1024, out_features=1024, bias=True)
          )
          (mlp): Qwen3VLVisionMLP(
            (linear_fc1): Linear(in_features=1024, out_features=4096, bias=True)
            (linear_fc2): Linear(in_features=4096, out_features=1024, bias=True)
            (act_fn): GELUTanh()
          )
        )
      )
 

In [ ]:
from tqdm import tqdm

In [ ]:
def process_data(df, question_type, output_file):

  if "model_output" not in df.columns:
    df["model_output"] = None
  if "output_probs" not in df.columns:
    df["output_probs"] = None

  for i in tqdm(df.index):
    if question_type == "binary":
      message = {"role": "user",
          "content": [
              {"type": "image", "url": df.at[i, "image_data_url"]},
              {"type": "text",
                "text": f"""In the picture {df.at[i, "description_for_answer"]}.
                Question 1: {df.at[i, "generated_question"]}.
                Answer with one word YES or NO.
                Question 2: How confident are you in this answer?
                Answer with one number 0-100, where 0 is “Absolutely not confident” and 100 is “Absolutely confident”.
                Return your answer in python list format[Question 1 answer, Question 2 answer]
                """}
          ]}
    elif question_type == "detail":
      message = {"role": "user",
         "content": [
             {"type": "image", "url": df.at[i, "image_data_url"]},
             {"type": "text",
              "text": f"""In the picture {df.at[i, "description_for_answer"]}.
              Question 1: {df.at[i, "generated_question"]}.
              Answer with one word.
              Question 2: How confident are you in this answer?
              Answer with one number 0-100, where 0 is “Absolutely not confident” and 100 is “Absolutely confident”.
              Return your answer in python list format[Question 1 answer, Question 2 answer]
              """}
         ]}

    inputs = processor.apply_chat_template(
      [message],
      add_generation_prompt=True,
      tokenize=True,
      return_dict=True,
      return_tensors="pt",
    ).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=40,
            return_dict_in_generate=True,  # возвращает объект с полями, не просто тензор
            output_scores=True,            # логиты на каждом шаге (после penalties)
        )

    # Декодирование ответа
    prompt_len = inputs["input_ids"].shape[-1]
    generated_ids = outputs.sequences[:, prompt_len:]
    output = processor.decode(generated_ids[0], skip_special_tokens=True)
    df.at[i, "model_output"] = output

    # Извлечение логитов / вероятностей
    probs_list = []
    for step, score in enumerate(outputs.scores):
        # score: [batch_size, vocab_size]
        probs = F.softmax(score[0].float(), dim=-1)
        top_probs, top_ids = probs.topk(5)
        #top_tokens = processor.tokenizer.convert_ids_to_tokens(top_ids.tolist())
        top_tokens = [
            processor.tokenizer.decode([tid], skip_special_tokens=False)
            for tid in top_ids.tolist()
        ]
        pairs = [(tok, f"{p:.6f}") for tok, p in zip(top_tokens, top_probs.tolist())]
        probs_list.append(pairs)

    df.at[i, "output_probs"] = probs_list

  df.to_csv(output_file, index=False)
  return df

In [ ]:
res = process_data(db_fd, "detail", "false_detail_results.csv")

/tmp/ipykernel_1240/774376844.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df["model_output"] = None
/tmp/ipykernel_1240/774376844.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df["output_probs"] = None
100%|██████████| 300/300 [39:20<00:00,  7.87s/it]


In [ ]:
res = process_data(db_con, "binary", "concurrence_results.csv")

/tmp/ipykernel_1240/774376844.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df["model_output"] = None
/tmp/ipykernel_1240/774376844.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df["output_probs"] = None
100%|██████████| 300/300 [40:08<00:00,  8.03s/it]


In [ ]:
res = process_data(db_i, "binary", "image_results.csv")

/tmp/ipykernel_1240/774376844.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df["model_output"] = None
/tmp/ipykernel_1240/774376844.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df["output_probs"] = None
100%|██████████| 300/300 [40:21<00:00,  8.07s/it]


In [ ]:
res = process_data(db_t, "binary", "text_results.csv")

In [ ]:
res

,tag,image_name,image_data_url,true_description,description_for_answer,description_for_question,false_source_image_name,generated_question,used_prompt,model_output,output_probs
2,image,882048099.jpg,"data:image/jpeg;base64,/9j/4AAQSkZJRgABAQEAtAC...",A woman in a blue bathing suit and straw hat ...,Four women in all black barefoot posing on th...,A woman in a blue bathing suit and straw hat ...,5922945248.jpg,Is there a woman in a blue bathing suit and st...,You need to formulate a single binary (yes/no)...,"[NO, 0]","[[([, 1.000000), (#, 0.000000), (!, 0.000000),..."
6,image,199413509.jpg,"data:image/jpeg;base64,/9j/4AAQSkZJRgABAQEAtAC...",A woman peers out from her halfway opened blu...,A man in a blue shirt smiles at a baby he 's ...,A woman peers out from her halfway opened blu...,787907341.jpg,Is there a woman peeking out from a blue door?,You need to formulate a single binary (yes/no)...,"[NO, 75]","[[([, 1.000000), (#, 0.000000), (!, 0.000000),..."
10,image,2825540754.jpg,"data:image/jpeg;base64,/9j/4AAQSkZJRgABAQEAtAC...",a little boy is running on the beach with a r...,A young man dressed in black dress clothes li...,a little boy is running on the beach with a r...,3823691082.jpg,Is there a little boy running on the beach wit...,You need to formulate a single binary (yes/no)...,"[NO, 100]","[[([, 1.000000), (#, 0.000000), (!, 0.000000),..."
14,image,4850944716.jpg,"data:image/jpeg;base64,/9j/4AAQSkZJRgABAQEAtAC...",an older man wearing a suit has his arms fold...,The little girl wearing a pink hat is bending...,an older man wearing a suit has his arms fold...,537579448.jpg,Is there an older man in a suit looking into a...,You need to formulate a single binary (yes/no)...,"[NO, 0]","[[([, 1.000000), (#, 0.000000), (!, 0.000000),..."
18,image,4526883901.jpg,"data:image/jpeg;base64,/9j/4AAQSkZJRgABAQEAtAC...","A white-haired man with glasses , wearing a g...",A woman in a brown UPS uniform is handling pa...,"A white-haired man with glasses , wearing a g...",3384702365.jpg,Is there a man working on a sculpture?,You need to formulate a single binary (yes/no)...,"[NO, 0]","[[([, 1.000000), (#, 0.000000), (!, 0.000000),..."
22,image,523249012.jpg,"data:image/jpeg;base64,/9j/4AAQSkZJRgABAQEAtAC...",The man in the white shirt is sitting on a ro...,a little boy is running on the beach with a r...,The man in the white shirt is sitting on a ro...,2825540754.jpg,Is there a man sitting on a rock with a pole i...,You need to formulate a single binary (yes/no)...,"[NO, 0]","[[([, 1.000000), (#, 0.000000), (!, 0.000000),..."
26,image,3183330562.jpg,"data:image/jpeg;base64,/9j/4AAQSkZJRgABAQEAtAC...",A group of kids in summer play clothes follow...,A blond woman in heels and a semi revealing d...,A group of kids in summer play clothes follow...,4541453490.jpg,Are there kids following a young man?,You need to formulate a single binary (yes/no)...,"[NO, 0]","[[([, 1.000000), (#, 0.000000), (!, 0.000000),..."
30,image,4812170955.jpg,"data:image/jpeg;base64,/9j/4AAQSkZJRgABAQEAtAC...",A small child in a blue outfit is looking off...,Two soldiers at an event center are watching ...,A small child in a blue outfit is looking off...,5450688421.jpg,Is there a child in a blue outfit looking at t...,You need to formulate a single binary (yes/no)...,"[NO, 0]","[[([, 1.000000), (#, 0.000000), (!, 0.000000),..."
34,image,523327429.jpg,"data:image/jpeg;base64,/9j/4AAQSkZJRgABAQEAtAC...",A man holding up a red sign that offers the h...,A golden retriever jumps off of a blue surfbo...,A man holding up a red sign that offers the h...,3648160673.jpg,Is there a man holding up a red sign?,You need to formulate a single binary (yes/no)...,"[NO, 0]","[[([, 1.000000), (#, 0.000000), (!, 0.000000),..."
38,image,2425148763.jpg,"data:image/jpeg;base64,/9j/4AAQSkZJRgABAQEAtAC...",A woman in a red shirt is speaking at a table...,A man with red-hair dressed in a reflective j...,A woman in a red shirt is speaking at a table...,3495815292.jpg,Is there a woman speakin

Запускаем модель через api (для false_detail), так как локально запустить модель qwen3-vl-8b-instruct не получилось из-за ограничений на gpu.

In [ ]:
import asyncio
from tqdm import tqdm
from openai import AsyncOpenAI

OPENROUTER_API_KEY = ""
MODEL = "qwen/qwen3-vl-8b-instruct"


async def process_df_async_detail(
    df,
    concurrency=8,
    save_every=5,
    save_path=None,
):
    df = df.copy()

    if "model_output" not in df.columns:
        df["model_output"] = None
    if "used_prompt" not in df.columns:
        df["used_prompt"] = None

    client = AsyncOpenAI(
        base_url="https://openrouter.ai/api/v1",
        api_key=OPENROUTER_API_KEY,
    )

    sem = asyncio.Semaphore(concurrency)

    async def process_one(index):
        async with sem:
            row = df.loc[index]

            content = [
                {
                    "type": "image_url",
                    "image_url": {"url": row["image_data_url"]},
                },
                {
                    "type": "text",
                    "text": f"""In the picture {row['description_for_answer']}.
                    Question 1: {row['generated_question']}.
                    Answer with one word.
                    Question 2: How confident are you in this answer?
                    Answer with one number 0-100, where 0 is "Absolutely not confident" and 100 is "Absolutely confident".
                    Return your answer in python list format [Question 1 answer, Question 2 answer].
                    """,
                },
            ]

            try:
                completion = await client.chat.completions.create(
                    model=MODEL,
                    messages=[
                        {
                            "role": "user",
                            "content": content,
                        }
                    ],
                )

                text = completion.choices[0].message.content.strip()
                used_prompt = content[1]["text"]
                return index, text, used_prompt

            except Exception as e:
                print(f"Ошибка в строке {index}: {e}")
                used_prompt = content[1]["text"]
                return index, None, used_prompt

    tasks = [process_one(i) for i in df.index]
    done_count = 0

    for fut in tqdm(asyncio.as_completed(tasks), total=len(tasks)):
        index, output, used_prompt = await fut

        df.at[index, "model_output"] = output
        df.at[index, "used_prompt"] = used_prompt
        done_count += 1

        if save_path and done_count % save_every == 0:
            df.to_csv(save_path, index=False)

    if save_path:
        df_to_save = df.drop(columns=["image_data_url"], errors="ignore")
        df_to_save.to_csv(save_path, index=False)

    return df

In [ ]:
df_questions = pd.read_csv("df_questions_new.csv")

db_fd = df_questions[df_questions["tag"] == "false_detail"]
result = await process_df_async_detail(
    db_fd,
    concurrency=8,
    save_path="fd_results_8b_api.csv"
)